# NumPy NN - NumPy Solutions

> **MLCourse · Data Science Foundations · 01_numpy**

Fully worked answers to `03_numpy_exercises`, mirrored task-for-task - every cell re-creates its
own inputs so you can jump straight to any number without scrolling back.

### How to use this notebook

- Attempt first! Reading a solution before struggling robs the exercise of its value.
- Each cell below states the task in one line, then solves it with comments explaining WHY.
- "Takeaway" notes highlight the transferable idea when there is one.

> 💡 **Pro tip:** diff your attempt against these cells mentally - different-but-correct is a
> win; same-result-different-tool usually means a performance or readability lesson is hiding.

In [1]:
import numpy as np                   # single dependency for all fifteen solutions

print("NumPy version:", np.__version__)

NumPy version: 2.4.6


### **Solution 1 - Array creation warm-up**

Build a 3x4 grid of 0..11, a 2x2 zero frame, five linspace points from -2 to 2, and eye(5).

In [2]:
grid = np.arange(12).reshape(3, 4)       # arange makes the values, reshape bends them into rows
empty_frame = np.zeros((2, 2))           # shape tuple -> all 0.0 floats by default
spread = np.linspace(-2, 2, 5)           # count-based: BOTH endpoints guaranteed (-2 and 2)
identity5 = np.eye(5)                    # ones on the diagonal, zeros elsewhere

print(grid)
print(empty_frame.shape, spread, identity5.shape)

# Takeaway: reshape never copies data here - arange's buffer is simply reinterpreted as 3x4.

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 2) [-2. -1.  0.  1.  2.] (5, 5)


### **Solution 2 - Slice like you mean it**

From `x = np.arange(30)`: last five elements, values 10..19, every third from index 4, reversed.

In [3]:
x = np.arange(30)                        # shared input

last_five = x[-5:]                       # negative start counts from the end
teens = x[10:20]                         # stop EXCLUSIVE -> include value 19 need stop 20
every_third_from_4 = x[4::3]             # empty start slot means "from index 4 to the end"
mirror = x[::-1]                         # negative step walks backwards over the whole array

print(last_five)
print(teens)
print(every_third_from_4)
print(mirror[:6], "...")

[25 26 27 28 29]
[10 11 12 13 14 15 16 17 18 19]
[ 4  7 10 13 16 19 22 25 28]
[29 28 27 26 25 24] ...


### **Solution 3 - Broadcasting calisthenics**

Add a length-3 vector to every row of a 4x3 matrix; center every column by its mean.

In [4]:
M = np.arange(12).reshape(4, 3)          # inputs recreated
v = np.array([10, 20, 30])

biased = M + v                           # v pads to (1, 3), then stretches down all 4 rows
centered = M - M.mean(axis=0, keepdims=True)   # keepdims -> (1, 3) means broadcast cleanly

print(biased)
print("column means after centering:", centered.mean(axis=0).round(9))

# Takeaway: axis=0 computes PER-COLUMN stats because it collapses rows; keepdims preserves the
# (1, 3) orientation so the subtraction broadcasts instead of raising or misaligning.

[[10 21 32]
 [13 24 35]
 [16 27 38]
 [19 30 41]]
column means after centering: [0. 0. 0.]


### **Solution 4 - Axis gymnastics on exam scores**

Per-student totals, per-exam averages, the flat max position, and each student's best exam index.

In [5]:
scores = np.array([[78, 92, 85, 67],
                   [91, 70, 88, 95],
                   [60, 75, 73, 81],
                   [89, 94, 90, 72],
                   [55, 63, 71, 68]])

student_totals = scores.sum(axis=1)      # collapse COLUMNS -> one total per row (per student)
exam_averages = scores.mean(axis=0)      # collapse ROWS    -> one average per column (per exam)
best_student_flat = scores.argmax()      # no axis -> argmax over the FLATTENED matrix
best_exam_per_student = scores.argmax(axis=1)   # within each row: which exam won

row, col = np.unravel_index(best_student_flat, scores.shape)   # bonus: decode the flat index
print(student_totals)
print(exam_averages.round(2))
print("flat:", best_student_flat, "-> (row, col):", (row, col))
print(best_exam_per_student)

[322 344 289 345 257]
[74.6 78.8 81.4 76.6]
flat: 7 -> (row, col): (np.int64(1), np.int64(3))
[1 3 3 1 2]


### **Solution 5 - Filter the thermometer log**

Count days above 25, list them, compute their share, and count the pleasant 18..26 band.

In [6]:
day_highs = np.array([18, 21, 25, 29, 31, 27, 22, 17, 30, 26])

hot_mask = day_highs > 25                          # boolean array, True where strictly hot
hot_count = int(hot_mask.sum())                    # Trues sum as 1 -> instant counting
hot_values = day_highs[hot_mask]                   # mask-indexing filters (and returns a COPY)
hot_share_pct = hot_count / day_highs.size * 100   # share against ALL entries
pleasant_count = int(((day_highs >= 18) & (day_highs <= 26)).sum())   # parenthesized & combine

print(hot_count, hot_values, hot_share_pct, pleasant_count)

# Takeaway: `.sum()` on a mask IS the count - no loops, no len(list(filter(...))).

5 [29 31 27 30 26] 50.0 5


### **Solution 6 - Label, grade, clamp**

Tall/short labels, letter grades via nested where, percentages clamped into [0, 100].

In [7]:
heights = np.array([165, 172, 181, 158, 190, 175])
exams = np.array([93, 77, 84, 59, 91])
raw_pct = np.array([-5.0, 12.0, 99.0, 130.0, 100.0, 47.0])

labels = np.where(heights >= 175, "tall", "short")            # elementwise if/else
grades = np.where(exams >= 90, "A",                # outermost test first...
                  np.where(exams >= 80, "B", "C"))  # ...fallback branch nests inside
clamped = np.clip(raw_pct, 0, 100)                            # both tails handled at once

print(labels)
print(grades)
print(clamped)

# Takeaway: np.where chains read inside-out like else-if ladders; np.clip beats writing two
# separate masks-and-replaces.

['short' 'short' 'tall' 'short' 'tall' 'tall']
['A' 'C' 'B' 'C' 'A']
[  0.  12.  99. 100. 100.  47.]


### **Solution 7 - Top-k leaderboard**

Indices and values of the k largest revenues, highest first - plus the partition shortcut noted.

In [8]:
revenue = np.array([120.5, 88.0, 310.2, 155.9, 275.0, 99.5])


def top_k(values, k):
    """Return (top_indices, top_values) for the k largest entries, highest first."""
    order = np.argsort(values)              # ascending permutation of positions
    top_indices = order[-k:][::-1]          # take the tail (largest), reverse -> descending
    top_values = values[top_indices]        # fancy-index to fetch matching amounts
    return top_indices, top_values


ti, tv = top_k(revenue, 3)
print(ti, tv)

fast_values_only = np.partition(revenue, len(revenue) - 3)[-3:]   # O(n): 3rd-largest placed,
print("partition route:", fast_values_only)                       # everything bigger sits right

# Takeaway: argsort when you need WHICH, partition when you only need WHAT - full sort is wasted
# work for top-k questions.

[2 4 3] [310.2 275.  155.9]
partition route: [155.9 275.  310.2]


### **Solution 8 - Z-score normalize by column**

Standardize every column of a feature matrix to mean 0 / std 1, then verify.

In [9]:
rng8 = np.random.default_rng(42)                              # identical data to the exercise
X_raw = rng8.normal(loc=[10, -5, 0, 200], scale=[1, 10, 0.5, 50], size=(8, 4))


def zscore_columns(X):
    """Return Z where every column of X has mean 0 and std 1 (ddof=0)."""
    mu = X.mean(axis=0, keepdims=True)       # (1, 4): one mean per column
    sigma = X.std(axis=0, keepdims=True)     # (1, 4): population std, NumPy default ddof=0
    return (X - mu) / sigma                  # broadcasting shifts+scales all rows uniformly


Z = zscore_columns(X_raw)
print("column means ~0 :", np.abs(Z.mean(axis=0)).round(6))
print("column stds ==1 :", Z.std(axis=0).round(6))

# Takeaway: this exact pattern (mean/std along axis 0, keepdims=True) is sklearn's
# StandardScaler in three lines - and the seed makes results reproducible across runs.

column means ~0 : [0. 0. 0. 0.]
column stds ==1 : [1. 1. 1. 1.]


### **Solution 9 - Moving average without loops**

Valid-mode moving average via the padded-cumsum trick (convolution variant shown commented).

In [10]:
prices = np.array([101., 103., 102., 105., 107., 106., 109., 111., 108., 110.,
                   113., 112.])


def moving_average(x, k=3):
    """Return the k-window moving average of x (valid mode)."""
    S = np.concatenate(([0.0], np.cumsum(x)))   # prepend 0 so differences align to windows
    window_sums = S[k:] - S[:-k]                # window i sum = cumulative[i+k] - cumulative[i]
    return window_sums / k


ma = moving_average(prices, 3)
print(ma.round(2))
print("length check:", ma.size == prices.size - 3 + 1)
print("first window check:", np.isclose(ma[0], prices[:3].mean()))

# Alternative one-liner: np.convolve(prices, np.ones(3) / 3, mode="valid") - same numbers.
# Takeaway: cumsum turns 'sum of any range' into ONE subtraction - O(n) total, loop-free.

[102.   103.33 104.67 106.   107.33 108.67 109.33 109.67 110.33 111.67]
length check: True
first window check: True


### **Solution 10 - One-hot encoding from labels**

Loop-free indicator matrix via one fancy-indexed scatter assignment.

In [11]:
labels = np.array([3, 0, 1, 3, 2, 1, 3])         # class id per sample


def one_hot(labels, n_classes=None):
    """Return an (n_samples, n_classes) int matrix of 0/1 indicators."""
    n_samples = labels.size
    if n_classes is None:
        n_classes = int(labels.max()) + 1        # infer from the data
    out = np.zeros((n_samples, n_classes), dtype=int)     # all-off canvas
    out[np.arange(n_samples), labels] = 1        # paired indices light up exactly one cell/row
    return out


H = one_hot(labels)
print(H)
print("one 1 per row?", np.array_equal(H.sum(axis=1), np.ones(labels.size, dtype=int)))

# Takeaway: out[rows, cols] pairs coordinates ELEMENTWISE - the standard trick for scatter
# updates (this is also exactly how embedding lookups work).

[[0 0 0 1]
 [1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [0 1 0 0]
 [0 0 0 1]]
one 1 per row? True


### **Solution 11 - Frequency table & compound filters**

Mode, divisibility-by-5 count, [10, 40) band count, and odd count from 100 seeded draws.

In [12]:
rng11 = np.random.default_rng(42)                # recreate the exact exercise data
draws = rng11.integers(0, 50, size=100)

uniq_vals, counts = np.unique(draws, return_counts=True)   # value -> frequency table
mode_value = uniq_vals[counts.argmax()]          # argmax finds the busiest bin
div_by_5 = int((draws % 5 == 0).sum())           # remainder test -> mask -> count
in_band = int(((draws >= 10) & (draws < 40)).sum())        # half-open interval needs BOTH tests
odd_count = int((draws % 2 == 1).sum())          # % works elementwise on integer arrays

print(mode_value, div_by_5, in_band, odd_count)  # expect: 4 11 63 49

4 11 63 49


### **Solution 12 - Pairwise distance matrix**

All N-by-N Euclidean distances in three vectorized lines using the broadcasting sandwich.

In [13]:
rng12 = np.random.default_rng(42)                # same six map points as the exercise
points = rng12.uniform(0, 10, size=(6, 2)).round(2)


def pairwise_distances(P):
    """Return the (N, N) matrix of Euclidean distances between all point pairs."""
    delta = P[:, None, :] - P[None, :, :]        # (N,1,D)-(1,N,D) -> (N,N,D) component diffs
    return np.sqrt((delta ** 2).sum(axis=-1))    # Pythagoras collapsed over the D-axis


Dm = pairwise_distances(points)
print(Dm.round(2))
print("shape/diag/symmetry:", Dm.shape,
      np.allclose(np.diag(Dm), 0), np.allclose(Dm, Dm.T))

# Takeaway: [:, None] vs [None, :] is THE pairwise pattern - it generalizes to cosine similarity,
# kernels and attention scores unchanged.

[[0.   2.72 8.66 3.47 6.46 6.33]
 [2.72 0.   8.14 1.32 7.72 5.39]
 [8.66 8.14 0.   6.94 5.27 2.81]
 [3.47 1.32 6.94 0.   7.17 4.15]
 [6.46 7.72 5.27 7.17 0.   5.35]
 [6.33 5.39 2.81 4.15 5.35 0.  ]]
shape/diag/symmetry: (6, 6) True True


### **Solution 13 - Image surgery (invert, crop, border)**

Photographic negative, center crop, and a zero-padded border canvas - no loops anywhere.

In [14]:
img = (np.arange(64) * 4).astype(np.uint8).reshape(8, 8)   # smooth gradient "image"

inverted = 255 - img                             # scalar broadcast; stays inside uint8 range
crop_center = img[2:6, 2:6]                      # plain slicing: view, no copy needed
canvas = np.zeros((12, 12), dtype=img.dtype)     # black canvas sized img + 2px border/frame
canvas[2:-2, 2:-2] = img                         # paste original into the interior
bordered = canvas

print(inverted[0, :4])                           # 255, 251, 247, 243 - darkest became brightest
print(crop_center.shape, bordered.shape)
print("corner border pixels are zero:", bordered[0, 0] == 0 and bordered[-1, -1] == 0)

# Takeaway: padding-by-allocation-then-slice-assign is the manual equivalent of np.pad - worth
# doing once by hand so np.pad feels obvious afterwards.

[255 251 247 243]
(4, 4) (12, 12)
corner border pixels are zero: True


### **Solution 14 - Solve the supply puzzle (linear system)**

Assemble Ax = b for `g + 2b = 2` and `3g - b = 13`, confirm solvability, solve, verify.

In [15]:
A_sys = np.array([[1.0, 2.0],
                  [3.0, -1.0]])                  # one ROW per invoice equation
b_vec = np.array([2.0, 13.0])

det = np.linalg.det(A_sys)                        # nonzero determinant <=> unique solution
if abs(det) < 1e-12:
    print("WARNING: singular system (det = 0) - no unique solution")
else:
    solution = np.linalg.solve(A_sys, b_vec)      # LU-backed direct solver (prefer over inv)
    print("det =", det, "| solution [g, b] =", solution)
    print("residual:", A_sys @ solution - b_vec, "| allclose:",
          np.allclose(A_sys @ solution, b_vec))   # substitute-back proof

# Takeaway: the solver lands on [g, b] = [4, -1] - when coefficients go negative, trust the
# algebra (and the residual check) over physical intuition. The det guard costs microseconds and
# catches degenerate systems that would otherwise surface as inf/nan far downstream.

det = -7.000000000000001 | solution [g, b] = [ 4. -1.]
residual: [4.4408921e-16 0.0000000e+00] | allclose: True


### **Solution 15 - Casino audit: two-dice Monte Carlo**

Roll two fair dice 100k times in one vectorized shot and audit the sums empirically.

In [16]:
n_rolls = 100_000
rng15 = np.random.default_rng(42)                 # same seed as the exercise -> same rolls


def dice_experiment(rng, n):
    """Return (sums, p_seven_empirical, freq_sums, freq_probs, p_ten_plus_empirical)."""
    rolls = rng.integers(1, 7, size=(n, 2))       # high EXCLUSIVE: faces 1..6 for BOTH dice
    sums = rolls.sum(axis=1)                      # collapse the pair axis -> n totals
    p_seven_empirical = float((sums == 7).mean()) # mean of a bool array = fraction of Trues
    freq_sums, freq_counts = np.unique(sums, return_counts=True)
    freq_probs = freq_counts / n                  # normalize counts into probabilities
    p_ten_plus_empirical = float((sums >= 10).mean())
    return sums, p_seven_empirical, freq_sums, freq_probs, p_ten_plus_empirical


sums, p7, fs, fp, p10 = dice_experiment(rng15, n_rolls)
print("P(sum==7)  empirical:", round(p7, 4), "| theory:", round(6 / 36, 4))
print("P(sum>=10) empirical:", round(p10, 4), "| theory:", round(6 / 36, 4))
for s, pr in zip(fs, fp):
    bar = "#" * int(pr * 300)                     # tiny ASCII histogram of the distribution
    print(f"{s:2d} {pr:.3f} {bar}")

# Takeaway: 100k samples land within ~0.002 of theory - Monte Carlo turns probability questions
# into counting exercises, and numpy does the counting in compiled C.

P(sum==7)  empirical: 0.1683 | theory: 0.1667
P(sum>=10) empirical: 0.1676 | theory: 0.1667
 2 0.028 ########
 3 0.055 ################
 4 0.083 ########################
 5 0.110 #################################
 6 0.139 #########################################
 7 0.168 ##################################################
 8 0.138 #########################################
 9 0.111 #################################
10 0.085 #########################
11 0.055 ################
12 0.028 ########


### Summary & key takeaways

- Creation (`arange+reshape`, `linspace`, `zeros`, `eye`) covers nearly every synthetic-input need.
- Slicing vocabulary (`start:stop:step`, negatives, `[::-1]`) replaces entire utility functions.
- Broadcasting + `keepdims=True` is the clean way to shift/scale rows or columns (Task 3, 8).
- Masks count (`sum`), filter (`arr[mask]`) and combine (`& | ~` with parentheses) in one breath.
- `argsort[-k:][::-1]` ranks; `partition` wins when order within top-k doesn't matter.
- Scatter assignments (`out[np.arange(n), labels] = 1`) build one-hot/embeddings without loops.
- The `[:, None] - [None, :]` sandwich delivers distance matrices in constant code.
- `np.linalg.solve` + a determinant guard is the professional habit for linear systems.
- Seeded Generators (`default_rng(42)`) make simulations reproducible AND auditable.